In [1]:
# Notebook 3 : Analyse et export pour Power BI
# Objectif : Analyser les données et préparer les exports finaux pour Power BI

# --- 1. Chargement des bibliothèques et des données ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Charger les tables dimensionnelles et de faits
dim_pays = pd.read_csv("data/processed/dim_pays.csv")
dim_annee = pd.read_csv("data/processed/dim_annee.csv")
dim_indicateur = pd.read_csv("data/processed/dim_indicateur.csv")
table_faits = pd.read_csv("data/processed/faits_indicateurs.csv")

print("✅ Données chargées avec succès !")

# --- 2. Analyse exploratoire ---

# Aperçu des données
print("\n--- Aperçu des tables ---")
print("Table dimension pays :")
print(dim_pays.head())
print("\nTable dimension année :")
print(dim_annee.head())
print("\nTable dimension indicateur :")
print(dim_indicateur.head())
print("\nTable de faits :")
print(table_faits.head())

# Statistiques descriptives
print("\n--- Statistiques descriptives ---")
print(table_faits.describe())

# Visualisation : Évolution de la population par région
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=table_faits[table_faits["indicator"] == "population_people"],
    x="year", y="value", hue="region",
    errorbar=None, estimator="mean"
)
plt.title("Évolution de la population par région")
plt.xlabel("Année")
plt.ylabel("Population (habitants)")
plt.xticks(rotation=45)
plt.legend(title="Région")
plt.tight_layout()
plt.show()

# Visualisation : Accès à l'eau potable par région
plt.figure(figsize=(12, 6))
sns.barplot(
    data=table_faits[table_faits["indicator"].isin(["basic_drinking_water_pct", "safely_managed_drinking_water_pct"])],
    x="region", y="value", hue="indicator",
    errorbar=None, estimator="mean"
)
plt.title("Accès à l'eau potable par région")
plt.xlabel("Région")
plt.ylabel("Pourcentage (%)")
plt.xticks(rotation=45)
plt.legend(title="Type d'accès")
plt.tight_layout()
plt.show()

# --- 3. Calcul des scores pondérés ---

# Fonction pour calculer un score pondéré
def calculer_score_pondere(df, weighted_columns):
    numerator = pd.Series(0.0, index=df.index)
    denominator = pd.Series(0.0, index=df.index)
    for column_name, weight in weighted_columns:
        valid_mask = df[column_name].notna()
        numerator = numerator + df[column_name].fillna(0) * weight
        denominator = denominator + valid_mask.astype(float) * weight
    return pd.Series(np.where(denominator > 0, numerator / denominator * 100, np.nan), index=df.index)

# Exemple : Calculer un score de vulnérabilité (à adapter selon vos besoins)
# Supposons que vous voulez pondérer :
# - Accès à l'eau potable (50%)
# - Stabilité politique (30%)
# - Mortalité liée à l'eau (20%)

# Extraire les données nécessaires pour 2016 (année commune)
df_2016 = table_faits[table_faits["year"] == 2016].copy()

# Pivoter pour avoir un DataFrame avec une ligne par pays et une colonne par indicateur
df_pivot = df_2016.pivot_table(
    index=["country", "region"],
    columns="indicator",
    values="value",
    aggfunc="first"
).reset_index()

# Calculer le score de vulnérabilité
weighted_columns = [
    ("basic_drinking_water_pct", 0.5),
    ("political_stability", 0.3),
    ("wash_mortality_rate_per_100k", -0.2)  # Inverser l'effet pour la mortalité
]

df_pivot["score_vulnerabilite"] = calculer_score_pondere(df_pivot, weighted_columns)

print("\n--- Score de vulnérabilité par pays (2016) ---")
print(df_pivot[["country", "region", "score_vulnerabilite"]].sort_values("score_vulnerabilite", ascending=False).head(10))

# Visualisation du score de vulnérabilité
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_pivot.sort_values("score_vulnerabilite", ascending=False).head(15),
    x="country", y="score_vulnerabilite", hue="region"
)
plt.title("Top 15 des pays par score de vulnérabilité (2016)")
plt.xlabel("Pays")
plt.ylabel("Score de vulnérabilité")
plt.xticks(rotation=90)
plt.legend(title="Région")
plt.tight_layout()
plt.show()

# --- 4. Export des résultats pour Power BI ---

# Exporter la table des scores
df_pivot.to_csv("data/processed/scores_vulnerabilite.csv", index=False)

# Exporter un résumé par région
resume_region = df_pivot.groupby("region").agg({
    "score_vulnerabilite": ["mean", "min", "max"],
    "country": "count"
}).reset_index()
resume_region.columns = ["region", "score_moyen", "score_min", "score_max", "nombre_pays"]
resume_region.to_csv("data/processed/resume_region.csv", index=False)

print("✅ Export des résultats d'analyse terminé !")

# --- 5. Documentation des exports ---
# Créer un fichier README.md pour expliquer les fichiers exportés
readme_content = """
# Documentation des exports pour Power BI

## Fichiers exportés

1. **dim_pays.csv** : Table dimensionnelle des pays et régions.
   - Colonnes : `country`, `region`

2. **dim_annee.csv** : Table dimensionnelle des années.
   - Colonnes : `year`

3. **dim_indicateur.csv** : Table dimensionnelle des indicateurs.
   - Colonnes : `indicator`, `unit`

4. **faits_indicateurs.csv** : Table de faits avec les valeurs des indicateurs.
   - Colonnes : `country`, `region`, `year`, `indicator`, `value`, `unit`, `source`, `scope_note`

5. **scores_vulnerabilite.csv** : Scores de vulnérabilité calculés pour 2016.
   - Colonnes : `country`, `region`, `basic_drinking_water_pct`, `political_stability`, `wash_mortality_rate_per_100k`, `score_vulnerabilite`

6. **resume_region.csv** : Résumé des scores par région.
   - Colonnes : `region`, `score_moyen`, `score_min`, `score_max`, `nombre_pays`

## Règles de nettoyage appliquées
- Les noms de pays ont été standardisés (suppression des accents, espaces, etc.).
- Les pays sans région ont été associés manuellement via `REGION_OVERRIDES`.
- Les valeurs de population ont été converties en habitants (multipliées par 1000).
- Les valeurs manquantes ont été supprimées pour les colonnes critiques (`country`, `year`, `indicator`).

## Unités et sources
- `population_people` : Nombre d'habitants (source : Population.csv)
- `political_stability` : Indice de stabilité politique (source : PoliticalStability.csv)
- `basic_drinking_water_pct` : Pourcentage d'accès à l'eau potable de base (source : BasicAndSafelyManagedDrinkingWaterServices.csv)
- `safely_managed_drinking_water_pct` : Pourcentage d'accès à l'eau potable gérée de manière sûre (source : BasicAndSafelyManagedDrinkingWaterServices.csv)
- `wash_mortality_rate_per_100k` : Taux de mortalité lié à l'eau et à l'assainissement pour 100 000 habitants (source : MortalityRateAttributedToWater.csv)
- `wash_deaths` : Nombre de décès liés à l'eau et à l'assainissement (source : MortalityRateAttributedToWater.csv)

## Structure recommandée pour Power BI
- Importer les fichiers CSV dans Power BI en utilisant les relations suivantes :
  - `faits_indicateurs[country]` → `dim_pays[country]`
  - `faits_indicateurs[region]` → `dim_pays[region]`
  - `faits_indicateurs[year]` → `dim_annee[year]`
  - `faits_indicateurs[indicator]` → `dim_indicateur[indicator]`
"""

with open("data/processed/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ Documentation des exports générée (README.md) !")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/dim_pays.csv'